# Partial Shape-Preserving (PSP) Splines — Interactive Demo

**Reference:** Q. Li, J. Tian, "Partial shape-preserving splines",
*Computer-Aided Design* **43** (2011) 394–409.

## Value proposition
| Property | B-spline | NURBS | **PSP spline** |
|---|---|---|---|
| Polynomial (non-rational) | ✓ | ✗ | **✓** |
| Partition of unity | ✓ | ✓ | **✓** |
| C^{n-1} smooth | ✓ | ✓ | **✓** |
| Basis reaches 1 (flat-top) | ✗ | via rational | **✓** |
| Weights without rational denominator | N/A | ✗ | **✓** |
| Extra δ dimension | ✗ | ✗ | **✓** |
| Selective interpolation | ✗ | ✗ | **✓** |


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
%matplotlib inline

from shape_blend_splines.basis import (
    smooth_unit_step, smooth_unit_step_delta, psp_basis, psp_partition,
    knots_from_weights, interpolated_indices, shape_preserving_interval
)
from shape_blend_splines.curve import (
    WeightedControlPolygonPSPSpline, HermitePSPSpline, BlendedPrimitivePSPSpline
)
print('PSP splines loaded.')


## 1. Smooth unit step H_n(x)  (Eq. 6)

The PSP basis is built from H_n, a smooth piecewise polynomial:
$$H_n(x) = \frac{1}{n!\,2^n} \sum_{k=0}^{n} (-1)^k \binom{n}{k} (x+n-2k)^n H_0(x+n-2k)$$


In [ ]:
x = np.linspace(-4, 4, 400)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, n in zip(axes, [1, 2, 3]):
    h = smooth_unit_step(x, n)
    ax.plot(x, h, 'steelblue', lw=2)
    ax.axhline(0.5, color='gray', lw=0.6, ls=':')
    ax.set_title(f'H_{n}(x)'); ax.set_ylim(-0.1, 1.1); ax.grid(alpha=0.2)
fig.suptitle('Smooth unit steps H_n(x)', y=1.02); plt.tight_layout(); plt.show()
print(f'H_3(0) = {smooth_unit_step(0.0, 3):.4f}  (expect 0.5)')
print(f'H_3(3) = {smooth_unit_step(3.0, 3):.4f}  (expect 1.0)')
print(f'H_3(-3) = {smooth_unit_step(-3.0, 3):.4f}  (expect 0.0)')


## 2. PSP basis B^{(n)}_{[a,b],δ}(x)  (Eq. 17)

$$B^{(n)}_{[a,b],\delta}(x) = H_{n,\delta}(x-a) - H_{n,\delta}(x-b)$$

The **flat-top** [a+δ, b−δ] is where B = 1 exactly (shape preservation).


In [ ]:
x = np.linspace(0, 8, 500)
a, b = 2.0, 6.0
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Vary delta
for delta, c in zip([0.1, 0.5, 1.0, 1.5, 1.9], plt.cm.plasma(np.linspace(0.1,0.9,5))):
    B = psp_basis(x, a, b, 3, delta)
    axes[0].plot(x, B, color=c, lw=2, label=f'δ={delta}')
    left, right = a+delta, b-delta
    if left < right:
        mask = (x>=left)&(x<=right)
        axes[0].fill_between(x[mask], 0, B[mask], color=c, alpha=0.15)
axes[0].axhline(1, color='k', lw=0.6, ls=':')
axes[0].set_title('B^(3)_[2,6],δ  for various δ  (gold=flat-top)')
axes[0].legend(fontsize=8); axes[0].set_ylim(-0.05, 1.12)

# Partition of unity
knots = np.array([0., 1., 2.5, 4., 5.])
x2 = np.linspace(-0.5, 5.5, 400)
B2 = psp_partition(x2, knots, 3, 0.4)
colors = plt.cm.tab10.colors
for i in range(B2.shape[0]):
    axes[1].plot(x2, B2[i], color=colors[i], lw=2, label=f'B_{i}')
axes[1].plot(x2, B2.sum(axis=0), 'k--', lw=1.5, label='Sum (POU)')
axes[1].set_title('Partition of unity over non-uniform knots (Eq. 18)')
axes[1].legend(fontsize=8); axes[1].set_ylim(-0.05, 1.15)
plt.tight_layout(); plt.show()


## 3. B-spline special case  (page 398)

When knots are equally spaced with unit spacing, the B-spline basis N_{i,n} equals
the PSP basis with a_i = i + n/2, δ = n/2.

For non-equal knots, B-spline and PSP differ.


In [ ]:
from shape_blend_splines.basis import bspline_basis
n_ctrl, degree = 5, 3
knots_bs = np.arange(n_ctrl + degree + 1, dtype=float)
t = np.linspace(float(degree), float(n_ctrl), 400)

fig, ax = plt.subplots(figsize=(9, 4))
colors = plt.cm.tab10.colors
for i in range(n_ctrl):
    Nip = bspline_basis(i, degree, t, knots_bs)
    delta_p = degree / 2.0  # = 1.5
    Bpsp = psp_basis(t, i + delta_p, i + delta_p + 1, degree, delta_p)
    c = colors[i % len(colors)]
    ax.plot(t, Nip, color=c, lw=2.5, label=f'N_{{{i},3}}')
    ax.plot(t, Bpsp, color=c, lw=1, ls='--')
ax.set_title('B-spline (solid) = PSP basis (dashed) for equal-spaced knots')
ax.legend(fontsize=8); ax.set_ylim(-0.05, 1.1); ax.grid(alpha=0.2); plt.show()

# Numerical verification
max_err = max(
    abs(bspline_basis(i, 3, t, knots_bs) - psp_basis(t, i+1.5, i+2.5, 3, 1.5)).max()
    for i in range(n_ctrl)
)
print(f'Max B-spline vs PSP error: {max_err:.2e}  (should be ~0)')


## 4. Weighted control-polygon design  (Eq. 21; Figs. 9, 10)

**Weights as knot spacings** (non-rational):
$$P(t) = \sum_{i=0}^{N} P_i\, B^{(n)}_{[a_i,a_{i+1}],\delta}(t), \quad w_i = a_{i+1}-a_i$$


In [ ]:
ctrl = np.array([[0,0],[1,1],[2,0],[3,1],[4,0]], dtype=float)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# (a) Vary weights
for w, lbl, c in [
    ([1,1,1,1,1], 'equal', 'steelblue'),
    ([1,3,1,1,1], 'P1 heavy', 'tomato'),
    ([1,1,1,3,1], 'P3 heavy', 'seagreen'),
]:
    spl = WeightedControlPolygonPSPSpline(ctrl, weights=w, n=3, delta=0.4)
    t = np.linspace(spl.knots[0], spl.knots[-1], 300)
    axes[0].plot(*spl.evaluate(t).T, color=c, lw=2, label=lbl)
axes[0].plot(*ctrl.T, 'o--', color='gray', lw=1, ms=5, alpha=0.5)
axes[0].set_title('(a) Vary weights (same δ=0.4)'); axes[0].legend(fontsize=8)
axes[0].set_aspect('equal'); axes[0].grid(alpha=0.15)

# (b) Vary delta (extra design dimension!)
for delta, c in zip([0.1, 0.4, 0.9], ['navy', 'steelblue', 'lightblue']):
    spl = WeightedControlPolygonPSPSpline(ctrl, n=3, delta=delta)
    t = np.linspace(spl.knots[0], spl.knots[-1], 300)
    axes[1].plot(*spl.evaluate(t).T, color=c, lw=2, label=f'δ={delta}')
axes[1].plot(*ctrl.T, 'o--', color='gray', lw=1, ms=5, alpha=0.5)
axes[1].set_title('(b) Same weights, vary δ (extra design dim.)'); axes[1].legend(fontsize=8)
axes[1].set_aspect('equal'); axes[1].grid(alpha=0.15)

# (c) Selective interpolation (long interval → flat-top)
weights_sel = [0.3, 2.5, 0.3, 2.5, 0.3]
spl_sel = WeightedControlPolygonPSPSpline(ctrl, weights=weights_sel, n=3, delta=0.4)
t = np.linspace(spl_sel.knots[0], spl_sel.knots[-1], 400)
pts = spl_sel.evaluate(t)
axes[2].plot(*pts.T, color='steelblue', lw=2, label='PSP curve')
axes[2].plot(*ctrl.T, 'o--', color='gray', lw=1, ms=5, alpha=0.5)
interp_idx = spl_sel.interpolated_control_points()
axes[2].scatter(*ctrl[interp_idx].T, s=80, color='tomato', zorder=5,
               label=f'Interpolated: {interp_idx}')
axes[2].set_title('(c) Selective interpolation (★ = exact)'); axes[2].legend(fontsize=8)
axes[2].set_aspect('equal'); axes[2].grid(alpha=0.15)

fig.suptitle('Weighted control-polygon PSP curves (Li & Tian 2011, Eq. 21)', y=1.02)
plt.tight_layout(); plt.show()


## 5. Fig. 11: Selective interpolation — two delta values

**Same control polygon + same weights + different δ → different curve.**
P0, P2, P5, P7 have long intervals; P1, P3, P4, P6 are short.


In [ ]:
CTRL = np.array([[0,0],[0.8,0.8],[1.8,0.4],[2.6,1.2],[3.4,0.2],[4.4,1.0],[5.2,0.3],[6.0,0.8]], dtype=float)
WEIGHTS = [2.0, 0.4, 2.0, 0.4, 0.4, 2.0, 0.4, 2.0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, delta in zip(axes, [1.0, 1.8]):
    spl = WeightedControlPolygonPSPSpline(CTRL, weights=WEIGHTS, n=3, delta=delta)
    t = np.linspace(spl.knots[0], spl.knots[-1], 600)
    pts = spl.evaluate(t)
    interp = spl.interpolated_control_points()
    
    ax.plot(*pts.T, 'steelblue', lw=2.5, label='PSP curve')
    ax.plot(*CTRL.T, 'o--', color='gray', lw=1, ms=5, alpha=0.5)
    ax.scatter(*CTRL[interp].T, s=80, color='tomato', zorder=5, label=f'Interpolated: {interp}')
    for i,(px,py) in enumerate(CTRL):
        ax.annotate(f'P{i}', (px,py), xytext=(3,5), textcoords='offset points', fontsize=8)
    ax.set_aspect('equal')
    ax.set_title(f'δ = {delta}  (interpolated: {interp})')
    ax.legend(fontsize=8); ax.grid(alpha=0.15)

fig.suptitle('Fig. 11: Same control polygon, different δ — extra design dimension\n(Li & Tian 2011)', y=1.02)
plt.tight_layout(); plt.show()


## 6. Hermite PSP spline  (Eq. 23)

Interpolate position P_i *and* velocity v_i at each node using the n=2 quadratic PSP basis.


In [ ]:
pts_h = np.array([[0,0],[1.5,1.5],[3,0],[4.5,1.5],[6,0]], dtype=float)
vel_h = np.array([[1,0.5],[0.5,-1],[1,0.5],[0.5,-1],[1,0.5]], dtype=float)
knots_h = np.array([0., 2., 4., 6., 8., 10.])
delta_h = 0.7

herm = HermitePSPSpline(pts_h, vel_h, knots=knots_h, delta=delta_h)
t_h = np.linspace(herm.knots[0], herm.knots[-1], 500)
curve_h = herm.evaluate(t_h)

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(*curve_h.T, 'steelblue', lw=2.5, label='Hermite PSP')
ax.scatter(*pts_h.T, s=60, color='tomato', zorder=5, label='Nodes P_i')
for i,(px,py) in enumerate(pts_h):
    vx,vy = vel_h[i]*0.3
    ax.annotate('', xytext=(px,py), xy=(px+vx,py+vy),
                arrowprops=dict(arrowstyle='->', color='seagreen', lw=2))
    ax.annotate(f'P{i}', (px,py), xytext=(4,6), textcoords='offset points', fontsize=9)
ax.set_aspect('equal'); ax.legend(fontsize=9); ax.grid(alpha=0.15)
ax.set_title(f'Hermite PSP: P(t_i)=P_i and P\'(t_i)=v_i exactly  (n=2, δ={delta_h})')
plt.tight_layout(); plt.show()

# Verify
for i, t_i in enumerate(herm.t_nodes):
    p = herm.evaluate(np.array([t_i]))[0]
    print(f'P({t_i:.1f}) = {p}  (P_{i} = {pts_h[i]})  error = {np.linalg.norm(p-pts_h[i]):.2e}')


## 7. Primitive blending  (Eq. 22)


In [ ]:
from shape_blend_splines.shapes import sine_wave, helix_2d
from functools import partial

def prim_sine(t):
    return np.column_stack([t * 2, np.sin(t * 2 * np.pi)])

def prim_line(t):
    return np.column_stack([t * 2, np.zeros_like(t)])

spl_blend = BlendedPrimitivePSPSpline(
    [prim_line, prim_sine, prim_line],
    knots=[0., 2., 4., 6.],
    n=3, delta=0.7
)
t_b = np.linspace(0, 6, 500)
pts_b = spl_blend.evaluate(t_b)

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(*pts_b.T, 'steelblue', lw=2.5, label='Blended PSP')
# Show primitives
for p, c, lbl in [(prim_line,'gray','line'), (prim_sine,'tomato','sine')]:
    ax.plot(*p(t_b).T, '--', color=c, lw=1.2, alpha=0.5, label=lbl)
ax.set_title('Primitive blending: sine embedded in flat lines (Eq. 22)')
ax.legend(fontsize=8); ax.grid(alpha=0.15); plt.tight_layout(); plt.show()
